In [ ]:
import boto3import osathena = boto3.client("athena", region_name=os.environ["AWS_REGION"])def lambda_handler(event, context):    query = """    SELECT user_token    FROM raw_logs    WHERE dt >= current_date - interval '2' day      AND event_ts >= (to_unixtime(current_timestamp) - 129600) * 1000    GROUP BY user_token    HAVING        SUM(CASE WHEN type = 'heartbeat' THEN 1 ELSE 0 END) > 0    AND SUM(CASE WHEN _activity <> 'unknown' THEN 1 ELSE 0 END) = 0    """    response = athena.start_query_execution(        QueryString=query,        QueryExecutionContext={"Database": "ameowzon_db"},        WorkGroup="primary",        ResultConfiguration={            "OutputLocation": "s3://ameowzon-athena-result-bucket/"        }    )    return response["QueryExecutionId"]